# 🧪 Thí nghiệm Stage 1 – So sánh Chiến lược Inference + YOLO26

**Mục tiêu:** Đánh giá 5 cấu hình trên tập val, bao gồm kiến trúc mới YOLO26s.

| TN | Model | Conf | SAHI | Mục đích |
|----|-------|------|------|----------|
| Baseline | YOLOv8s | 0.25 | ❌ | Mốc so sánh |
| TN1 | YOLOv8s | 0.15 | ❌ | Đo tác động hạ confidence |
| TN2 | YOLOv8s | 0.25 | ✅ | Đo tác động SAHI cho vật nhỏ |
| TN3 | YOLOv8s | 0.15 | ✅ | Kết hợp cả hai |
| TN4 | **YOLO26s** | 0.25 | ❌ | So sánh kiến trúc mới nhất 2026 |

⚠️ **Lưu ý:** TN4 sẽ **tự động train YOLO26s** trên cùng dataset binary (cùng siêu tham số với YOLOv8s) để so sánh công bằng. Thời gian ước tính: ~30-60 phút (GPU P100).

**Yêu cầu:** GPU accelerator + Internet enabled

In [ ]:
# Cell 1: Clone repo + Cài đặt
!git clone https://github.com/Shiba-dotcom/waste-detection2-Stage.git
!pip install -q ultralytics sahi

In [ ]:
# Cell 2: Chuẩn bị dữ liệu
import os, shutil

!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/TrashNet
!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/RealWaste
!mkdir -p /kaggle/working/waste-detection2-Stage/data/raw

datasets_to_copy = [
    {"src": "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/TrashNet"},
    {"src": "/kaggle/input/datasets/sohamchaudhari2004/taco-trash-detection-dataset/data",
     "dst": "/kaggle/working/waste-detection2-Stage/data/raw"},
    {"src": "/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/RealWaste"}
]

for task in datasets_to_copy:
    if os.path.exists(task["src"]):
        os.makedirs(task["dst"], exist_ok=True)
        shutil.copytree(task["src"], task["dst"], dirs_exist_ok=True)
        print(f"OK: {os.path.basename(task['src'])} -> {task['dst']}")
    else:
        print(f"SKIP: {task['src']}")

print("\nDone.")

In [ ]:
# Cell 3: Tiền xử lý dữ liệu
%cd /kaggle/working/waste-detection2-Stage
!python src/data_prep/data_cleaning.py
!python src/Training_dataYolo.py
!python src/data_prep/split_dataset.py
print("\n✅ Dữ liệu binary đã sẵn sàng!")

In [ ]:
# Cell 4: Copy model YOLOv8s đã train (stage1_best.pt)
# Đảm bảo file stage1_best.pt tồn tại trong thư mục models/
import shutil
from pathlib import Path

project = Path("/kaggle/working/waste-detection2-Stage")
best_pt = project / "stage1_best.pt"
yolo_best = project / "yolo_runs" / "stage1_binary_yolov8s" / "weights" / "best.pt"

if not best_pt.exists() and not yolo_best.exists():
    print("[WARN] Không tìm thấy stage1_best.pt!")
    print("       Cần train YOLOv8s trước hoặc upload file weights.")
    print("       Script sẽ báo lỗi ở bước tiếp theo.")
else:
    print(f"OK: Tìm thấy weights tại: {best_pt if best_pt.exists() else yolo_best}")

In [ ]:
# Cell 5: 🚀 Chạy toàn bộ thí nghiệm (bao gồm train YOLO26s + đánh giá 5 TN)
# ⏱️ Ước tính: 30-60 phút (train YOLO26s) + 10-15 phút (chạy 5 thí nghiệm)
%cd /kaggle/working/waste-detection2-Stage
!python notebooks/experiment_stage1_inference.py

In [ ]:
# Cell 6: Hiển thị kết quả
import pandas as pd
from IPython.display import display, HTML
from pathlib import Path

result_dir = Path("/kaggle/working/waste-detection2-Stage/results/stage1_experiments")

print("="*70)
print("  📊 BẢNG SO SÁNH TỔNG HỢP (5 Thí nghiệm)")
print("="*70)
df = pd.read_csv(result_dir / "comparison_table.csv")
display(df.style.highlight_max(subset=["Precision", "Recall", "F1"], color="lightgreen"))

print("\n" + "="*70)
print("  📈 CẢI THIỆN SO VỚI BASELINE")
print("="*70)
df_delta = pd.read_csv(result_dir / "delta_vs_baseline.csv")
display(df_delta.style.applymap(lambda v: 'color: green' if isinstance(v, (int, float)) and v > 0 else ('color: red' if isinstance(v, (int, float)) and v < 0 else '')))

print("\n" + "="*70)
print("  📏 RECALL THEO KÍCH THƯỚC VẬT THỂ")
print("="*70)
df_size = pd.read_csv(result_dir / "size_analysis.csv")
pivot = df_size.pivot_table(index="Thí nghiệm", columns="Kích thước", values="Recall", aggfunc="first")
display(pivot.style.highlight_max(color="lightgreen"))

In [ ]:
# Cell 7: Hiển thị biểu đồ
from IPython.display import Image as IPImage, display
from pathlib import Path

chart = Path("/kaggle/working/waste-detection2-Stage/results/stage1_experiments/experiments_comparison.png")
if chart.exists():
    display(IPImage(filename=str(chart), width=1200))
else:
    print("[WARN] Không tìm thấy biểu đồ")

In [ ]:
# Cell 8: Xem kết quả training YOLO26s (nếu có)
from pathlib import Path
from IPython.display import Image as IPImage, display

yolo26_dir = Path("/kaggle/working/waste-detection2-Stage/results/yolo26_runs/stage1_yolo26s")

print("📊 YOLO26s Training Results")
print("="*50)

# Hiển thị training curves
for img_name in ["results.png", "confusion_matrix.png", "F1_curve.png", "PR_curve.png"]:
    img_path = yolo26_dir / img_name
    if img_path.exists():
        print(f"\n--- {img_name} ---")
        display(IPImage(filename=str(img_path), width=800))

# Hiển thị results.csv
import pandas as pd
csv_path = yolo26_dir / "results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    print("\n--- Metrics cuối cùng (epoch cuối) ---")
    last = df.iloc[-1]
    for col in ["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
        if col in df.columns:
            print(f"  {col}: {last[col]:.4f}")

In [ ]:
# Cell 9: Download kết quả
import shutil

# Nén toàn bộ kết quả
result_dir = "/kaggle/working/waste-detection2-Stage/results"
shutil.make_archive("/kaggle/working/stage1_all_experiments", 'zip', result_dir)
print("📦 Đã nén: /kaggle/working/stage1_all_experiments.zip")
print("   Bao gồm:")
print("   - stage1_experiments/ (bảng so sánh, biểu đồ, JSON)")
print("   - yolo26_runs/stage1_yolo26s/ (training logs, weights YOLO26)")
print("   → Download file này về máy để lưu trữ")